# Many Models Forecasting Demo

This notebook showcases how to run MMF on a serverless compute using MLForecast-based global models (`MLForecastLGBM`, `MLForecastAutoLGBM`). We will use [M4 competition](https://www.sciencedirect.com/science/article/pii/S0169207019301128#sec5) monthly data.

Unlike [`global_serverless_dl.ipynb`](https://github.com/databricks-industry-solutions/many-model-forecasting/blob/main/examples/serverless/global_serverless_dl.ipynb) (GPU serverless + NeuralForecast), this notebook runs on plain CPU serverless — LightGBM is CPU-native and benefits from serverless's quick startup.

**Note that for a large scale forecasting use case (number of time series greater than 100), we recommend using a non-serverless compute, preferably a single-node CPU cluster with [Databricks Runtime 18 for ML](https://docs.databricks.com/aws/en/release-notes/runtime/18ml).**

### Serverless Compute setup

Attach this notebook to a [serverless compute](https://docs.databricks.com/aws/en/compute/serverless/notebooks). Then go to the [Configuration](https://docs.databricks.com/aws/en/compute/serverless/dependencies) tab and set the [Environment version](https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version) to Standard v5. In the Dependencies section, [add the path](https://docs.databricks.com/aws/en/compute/serverless/dependencies#create-common-utilities-to-share-across-your-workspace) to your `many-model-forecasting` directory: e.g., `/Workspace/Users/ryuta.yoshimatsu@databricks.com/many-model-forecasting`. This is required to use the MMF functions within the notebooks.

### Install and import packages

In [ ]:
%pip install -r ../../requirements-global.txt --quiet
%pip install datasetsforecast==0.0.8 --quiet
dbutils.library.restartPython()

In [ ]:
import logging
import pathlib
import pandas as pd
from datasetsforecast.m4 import M4
from mmf_sa import run_forecast

# Suppress MLflow context resolution warnings on serverless compute
logging.getLogger("mlflow.tracking.context.registry").setLevel(logging.ERROR)

# Suppress py4j verbose logging
logging.getLogger("py4j.clientserver").setLevel(logging.WARNING)
logging.getLogger("py4j.java_gateway").setLevel(logging.WARNING)

### Prepare data

In [ ]:
# Number of time series
n = 100


def create_m4_monthly():
    y_df, _, _ = M4.load(directory=str(pathlib.Path.home()), group="Monthly")
    target_ids = {f"M{i}" for i in range(1, n)}
    y_df = y_df[y_df["unique_id"].isin(target_ids)]
    y_df = (
        y_df.groupby("unique_id", group_keys=False)
             .apply(lambda g: transform_group(g, g.name))
             .reset_index(drop=True)
    )
    return y_df


def transform_group(df, unique_id):
    if len(df) > 60:
        df = df.iloc[-60:]
    start = pd.Timestamp("2018-01-01")
    date_idx = pd.date_range(start=start, periods=len(df), freq="ME", name="ds")
    res_df = pd.DataFrame({
        "ds": date_idx,
        "unique_id": unique_id,
        "y": df["y"].to_numpy()
    })
    return res_df

In [ ]:
catalog = "mmf"
db = "m4"
user = spark.sql('select current_user() as user').collect()[0]['user']

In [ ]:
_ = spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
_ = spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{db}")

(
    spark.createDataFrame(create_m4_monthly())
    .write.format("delta").mode("overwrite")
    .saveAsTable(f"{catalog}.{db}.serverless_train")
)

In [ ]:
display(
  spark.sql(f"select * from {catalog}.{db}.serverless_train where unique_id in ('M1', 'M2', 'M3', 'M4', 'M5') order by unique_id, ds")
  )

### Models

In [ ]:
active_models = [
    "MLForecastLGBM",
    "MLForecastAutoLGBM",
]

### Run MMF

In [ ]:
run_forecast(
    spark=spark,
    train_data=f"{catalog}.{db}.serverless_train",
    scoring_data=f"{catalog}.{db}.serverless_train",
    scoring_output=f"{catalog}.{db}.serverless_scoring_output",
    evaluation_output=f"{catalog}.{db}.serverless_evaluation_output",
    model_output=f"{catalog}.{db}",
    group_id="unique_id",
    date_col="ds",
    target="y",
    freq="M",
    prediction_length=3,
    backtest_length=12,
    stride=1,
    metric="smape",
    train_predict_ratio=1,
    data_quality_check=True,
    resample=False,
    active_models=active_models,
    experiment_path=f"/Users/{user}/mmf/serverless",
    use_case_name="serverless",
    accelerator="cpu",
)

### Evaluate

In [ ]:
display(spark.sql(f"""
    select * from {catalog}.{db}.serverless_evaluation_output
    where unique_id = 'M1' and model in ('MLForecastLGBM', 'MLForecastAutoLGBM')
    order by unique_id, model, backtest_window_start_date
    """))

### Forecast

In [ ]:
display(spark.sql(f"""
    select * from {catalog}.{db}.serverless_scoring_output
    where unique_id = 'M1' and model in ('MLForecastLGBM', 'MLForecastAutoLGBM')
    order by unique_id, model, ds
    """))

### Delete Tables

In [ ]:
#display(spark.sql(f"delete from {catalog}.{db}.serverless_evaluation_output where model in ('MLForecastLGBM', 'MLForecastAutoLGBM')"))

In [ ]:
#display(spark.sql(f"delete from {catalog}.{db}.serverless_scoring_output where model in ('MLForecastLGBM', 'MLForecastAutoLGBM')"))